# Air Quality Inequality Dashboard

**Project Goal:** This dashboard explores the historical relationships between county-level air quality, median household income, and lung cancer incidence rates.

### What you should learn from this dashboard:
**Tab 1: Health Focus**
* Does a higher percentage of unhealthy air days correlate with higher lung cancer incidence rates? 
* How to use: Drag the **Min Median AQI Slider** to observe how the cancer distribution shifts in under different air quality index.

**Tab 2: Economic Focus**
* Do lower income counties experience worse air quality compared to wealthier counties?
* How to use: Use the **Income Tier Dropdown** to highlight specific economic groups and see where they fall on the wealth-gap box plot.

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load analysis-ready data
conn = sqlite3.connect('aqi_project.db')
df = pd.read_sql("SELECT * FROM analysis_data", conn)
conn.close()


aqi_slider = widgets.IntSlider(
    min=int(df['median_aqi'].min()), 
    max=int(df['median_aqi'].max()), 
    step=5, 
    value=int(df['median_aqi'].min()),
    description='Min Median AQI:',
    style={'description_width': 'initial'}
)

tier_options = ['All Tiers'] + list(df['income_tier'].dropna().unique())
dropdown_tier = widgets.Dropdown(
    options=tier_options,
    value='All Tiers',
    description='Highlight Income Tier:',
    style={'description_width': 'initial'}
)

health_output = widgets.Output()
economy_output = widgets.Output()


def update_health_dashboard(change=None):
    with health_output:
        clear_output(wait=True)
        min_aqi = aqi_slider.value
        filtered_df = df[df['median_aqi'] >= min_aqi]
        
        # Create the figure object
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        sns.regplot(
            data=filtered_df, x='pct_unhealthy_air', y='incidence_rate', 
            ax=axes[0], scatter_kws={'alpha': 0.5, 'color': '#EF553B'}, line_kws={'color': 'darkred'}
        )
        axes[0].set_title(f'Air Quality vs. Cancer (AQI >= {min_aqi})', fontweight='bold')
        axes[0].set_xlabel('Percentage of Unhealthy Air Days (%)')
        axes[0].set_ylabel('Lung Cancer Incidence Rate (per 100k)')
        
        sns.histplot(data=filtered_df, x='incidence_rate', kde=True, ax=axes[1], color='purple')
        axes[1].set_title(f'Distribution of Cancer Rates (AQI >= {min_aqi})', fontweight='bold')
        axes[1].set_xlabel('Lung Cancer Incidence Rate (per 100k)')
        axes[1].set_ylabel('Number of Counties')
        
        plt.tight_layout()
        
        display(fig)       
        plt.close(fig)     

def update_economy_dashboard(change=None):
    with economy_output:
        clear_output(wait=True)
        selected_tier = dropdown_tier.value
        
        # Create the figure object
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        if selected_tier == 'All Tiers':
            plot_df = df
            title_suffix = "(All Counties)"
        else:
            plot_df = df[df['income_tier'] == selected_tier]
            title_suffix = f"({selected_tier} Only)"
            
        sns.regplot(
            data=plot_df, x='median_income', y='pct_unhealthy_air', 
            ax=axes[0], scatter_kws={'alpha': 0.5, 'color': '#00CC96'}, line_kws={'color': 'darkblue'}
        )
        axes[0].set_title(f'Income vs. Air Quality {title_suffix}', fontweight='bold')
        axes[0].set_xlabel('Median Household Income ($)')
        axes[0].set_ylabel('Percentage of Unhealthy Air Days (%)')

        tier_order = ['Low', 'Med-Low', 'Med-High', 'High']
        sns.boxplot(
            data=df, x='income_tier', y='pct_unhealthy_air', 
            hue='income_tier', legend=False, 
            ax=axes[1], order=tier_order, palette='viridis'
        )
        axes[1].set_title('Air Quality Distribution across Income Brackets', fontweight='bold')
        axes[1].set_xlabel('Income Tier')
        axes[1].set_ylabel('Percentage of Unhealthy Air Days (%)')
        

        if selected_tier != 'All Tiers':
            idx = tier_order.index(selected_tier)
            
            # Use axvspan to draw a vertical rectangle in the background
            axes[1].axvspan(idx - 0.5, idx + 0.5, color="skyblue", alpha=0.15, zorder=0)

        plt.tight_layout()
        
        display(fig)      
        plt.close(fig)   

# Display dashbard
aqi_slider.observe(update_health_dashboard, names='value')
dropdown_tier.observe(update_economy_dashboard, names='value')

update_health_dashboard()
update_economy_dashboard()

tab_health = widgets.VBox([aqi_slider, health_output])
tab_economy = widgets.VBox([dropdown_tier, economy_output])

dashboard = widgets.Tab(children=[tab_health, tab_economy])
dashboard.set_title(0, 'AQI and Cancer')
dashboard.set_title(1, 'AQI and Income')

display(dashboard)

<Figure size 640x480 with 0 Axes>